# 顔収集・LoRA学習・生成検証

Jetsonコンテナ内で上から順に実行します。冒頭の対象指定は顔収集、学習、生成検証のすべてに反映されます。対象の最終重みが揃っていれば顔収集と学習を自動で飛ばします。

1. `bash ./docker/run_l4t.sh` でコンテナを起動
2. 下の設定セルでキャラクター・項目・検証条件を指定
3. セルを上から実行し、最後の比較画像とJSON/CSVを確認

学習は `train_lora.py`、検証は `validate_lora.py` を呼ぶため、CLIとNotebookの処理内容は共通です。

## 編集する設定

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("./dataset")
TARGET_CHARACTERS = []  # []なら全キャラ。例: ["character-a"]
TARGET_FOLDERS = []  # []なら全項目。lora/portrait/anime/game/illust/face
RUN_FACE_COLLECTION = True
RUN_TRAINING = True
RUN_VALIDATION = True
FORCE_RETRAIN = False
REBUILD_FACES = False
VALIDATION_SCALES = [0.6, 0.8, 1.0]
VALIDATION_SEEDS = [42]
VALIDATION_STEPS = 25
VALIDATION_GUIDANCE_SCALE = 7.0
VALIDATE_CHECKPOINTS = False
VALIDATION_LOCAL_FILES_ONLY = False
VALIDATION_CPU_OFFLOAD = False

FOLDERS = ["lora", "portrait", "anime", "game", "illust", "face"]
selected_folders = TARGET_FOLDERS or FOLDERS
unknown = sorted(set(selected_folders) - set(FOLDERS))
if unknown:
    raise ValueError(f"不明な項目: {unknown}")


## 環境・対象と入力画像の確認

In [ ]:
import gc, json, subprocess, sys
from datetime import datetime
import matplotlib.pyplot as plt
import torch
from PIL import Image
from safetensors import safe_open

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
def list_images(path):
    path = Path(path)
    return sorted(p for p in path.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS) if path.is_dir() else []

if not DATASET_ROOT.is_dir():
    raise RuntimeError(f"データセットがありません: {DATASET_ROOT}")
all_characters = {p.name: p for p in sorted(DATASET_ROOT.iterdir()) if p.is_dir() and any((p / f).is_dir() for f in FOLDERS)}
missing_names = set(TARGET_CHARACTERS) - set(all_characters)
if missing_names:
    raise RuntimeError(f"存在しないキャラクター: {sorted(missing_names)}")
characters = ({name: all_characters[name] for name in TARGET_CHARACTERS} if TARGET_CHARACTERS else all_characters)
print("Python:", sys.version.split()[0], "Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("対象キャラ:", list(characters), "対象項目:", selected_folders)
for required in map(Path, ["anime_face_collect.py", "train_lora.py", "validate_lora.py"]):
    if not required.is_file(): raise RuntimeError(f"必要なスクリプトがありません: {required}")

preview = []
for character, root in characters.items():
    print(f"\n{character}")
    for folder in selected_folders:
        images = list_images(root / folder)
        caption = root / folder / "_common_caption.txt"
        weight = root / "folder_loras" / f"{character}-{folder}.safetensors"
        print(f"  {folder}: images={len(images)}, caption={caption.is_file()}, weight={weight.is_file()}")
        if caption.is_file(): print("   ", caption.read_text(encoding="utf-8", errors="replace").strip()[:240])
        if images and len(preview) < 12: preview.append((f"{character}/{folder}", images[0]))
if preview:
    cols, rows = min(4, len(preview)), (len(preview) + min(4, len(preview)) - 1) // min(4, len(preview))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = list(getattr(axes, "flat", [axes]))
    for ax in axes: ax.axis("off")
    for ax, (label, path) in zip(axes, preview):
        ax.imshow(Image.open(path).convert("RGB")); ax.set_title(label); ax.axis("off")
    plt.tight_layout(); plt.show()


## 顔収集・学習

対象の最終重みがすべて正常なら両方をスキップします。不足時だけ `anime_face_collect.py` を隔離Python環境で実行し、続いて現行 `train_lora.py` を呼びます。

In [ ]:
def valid_weight(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0: return False
    try:
        with safe_open(path, framework="pt", device="cpu") as stream: return bool(list(stream.keys()))
    except Exception as exc:
        print(f"無効な重み: {path}: {exc}"); return False

missing_weights = [(name, folder, root / "folder_loras" / f"{name}-{folder}.safetensors") for name, root in characters.items() for folder in selected_folders if FORCE_RETRAIN or not valid_weight(root / "folder_loras" / f"{name}-{folder}.safetensors")]
if not missing_weights:
    print("対象の最終重みがすべて存在します。顔収集と学習をスキップします。")
else:
    print("学習が必要:", [(c, f) for c, f, _ in missing_weights])
    if DATASET_ROOT.resolve() != Path("./dataset").resolve():
        raise RuntimeError("顔収集・学習時は各スクリプトのDATASET_ROOTと一致させてください")
    missing_faces = [name for name, root in characters.items() if not list_images(root / "face")]
    if missing_faces and RUN_FACE_COLLECTION:
        cmd = [sys.executable, "anime_face_collect.py", "--characters", *missing_faces]
        if REBUILD_FACES: cmd.append("--rebuild")
        print("顔収集:", " ".join(cmd)); subprocess.run(cmd, check=True)
    elif missing_faces: print("face画像不足（収集無効）:", missing_faces)
    if RUN_TRAINING:
        cmd = [sys.executable, "train_lora.py", "--train"]
        if TARGET_CHARACTERS: cmd += ["--characters", *TARGET_CHARACTERS]
        if TARGET_FOLDERS: cmd += ["--folders", *TARGET_FOLDERS]
        if FORCE_RETRAIN: cmd.append("--force")
        print("学習:", " ".join(cmd)); subprocess.run(cmd, check=True)
    else: print("RUN_TRAINING=Falseのため学習しません")
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 学習済み重みのmetadata

In [ ]:
for character, root in characters.items():
    for folder in selected_folders:
        path = root / "folder_loras" / f"{character}-{folder}.safetensors"
        if not valid_weight(path): print(f"[missing] {character}/{folder}"); continue
        with safe_open(path, framework="pt", device="cpu") as stream: metadata = stream.metadata() or {}
        print(f"\n{character}/{folder}: {path} ({path.stat().st_size / 1024**2:.2f} MB)")
        for key in ["ss_sd_model_name", "ss_steps", "ss_max_train_steps", "ss_network_dim", "ss_network_alpha", "ss_mixed_precision"]:
            if key in metadata: print(f"  {key}: {metadata[key]}")


## 生成検証

同じseed・条件でベースラインとLoRA強度を比較します。結果は `ai-image-lab-work/output/lora_validation/` に画像、比較グリッド、`results.json`、`results.csv` として保存されます。途中重みも比較する場合は冒頭を `VALIDATE_CHECKPOINTS=True` にします。

In [ ]:
VALIDATION_RUN_DIR = None
if RUN_VALIDATION:
    run_name = "notebook_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    cmd = [sys.executable, "validate_lora.py", "--dataset-root", str(DATASET_ROOT), "--run-name", run_name, "--steps", str(VALIDATION_STEPS), "--guidance-scale", str(VALIDATION_GUIDANCE_SCALE), "--scales", *map(str, VALIDATION_SCALES), "--seeds", *map(str, VALIDATION_SEEDS)]
    if TARGET_CHARACTERS: cmd += ["--characters", *TARGET_CHARACTERS]
    if TARGET_FOLDERS: cmd += ["--folders", *TARGET_FOLDERS]
    if VALIDATE_CHECKPOINTS: cmd.append("--checkpoints")
    if VALIDATION_LOCAL_FILES_ONLY: cmd.append("--local-files-only")
    if VALIDATION_CPU_OFFLOAD: cmd.append("--cpu-offload")
    print("検証:", " ".join(cmd)); subprocess.run(cmd, check=True)
    VALIDATION_RUN_DIR = Path("ai-image-lab-work/output/lora_validation") / run_name
    print("検証結果:", VALIDATION_RUN_DIR.resolve())
else: print("RUN_VALIDATION=Falseのため検証しません")


## 検証結果の表示

In [ ]:
if VALIDATION_RUN_DIR and VALIDATION_RUN_DIR.is_dir():
    report = json.loads((VALIDATION_RUN_DIR / "results.json").read_text(encoding="utf-8"))
    errors = [item for item in report if item.get("status") != "ok"]
    print(f"成功={len(report) - len(errors)}, missing/error={len(errors)}")
    for item in errors: print(item)
    summary = VALIDATION_RUN_DIR / "grids" / "all_characters_summary.png"
    if summary.is_file(): display(Image.open(summary))
    for character in characters:
        path = VALIDATION_RUN_DIR / character / "grids" / "summary.png"
        if path.is_file(): print(character); display(Image.open(path))
